In [1]:
# !pip install --upgrade openai pydantic

In [2]:
import openai
import os
from pydantic import BaseModel
from typing import List

from gm_instruction_generator import GM_InstructionGenerator
from gm_json_generator import GM_JSONGenerator
from vm_logic_validator import VM_LogicValidator
from gm_sample_picker import SamplePicker
from gm_system_prompts import SystemPromptRepository

# Create an instance of SamplePicker, assuming your seed files are stored in the "seed" folder.
sample_picker = SamplePicker(seed_folder="seed")

# For example, request 3 samples from the "easy" type.
seed_type="easy"
data_sample = sample_picker.get_full_examples(seed_type=seed_type, num_examples=3)

#print(data_sample)

if data_sample is None:
    print(f"Not enough unchosen samples available from '{seed_type}' seeds.")
    sample_instructions = []
    sample_jsons = []
else:
    sample_instructions, sample_jsons = (list(x) for x in zip(*[(ex["instruction"], ex["json"]) for ex in data_sample["examples"]]))

os.environ['OPENAI_API_KEY'] = '***REMOVED***'

# Create an instance of the InstructionGenerator
instruction_generator = GM_InstructionGenerator()
instruction_generation_prompt = SystemPromptRepository.get_instruction_generation_prompt()

# Create an instance of the JSONGenerator
json_generator = GM_JSONGenerator()
json_generation_prompt = SystemPromptRepository.get_json_generation_prompt()

# Create an instance of the LogicValidator
logic_validator = VM_LogicValidator()
logic_validation_prompt = SystemPromptRepository.get_logic_validation_prompt()



# Generate new instructions (e.g., 20 new instructions)
# Arguments:
# 1. sample instructions to include in the system prompt
# 2. instruction_generatoion system prompt
# 3. number of newly generated instructions
new_instructions, difficulty = instruction_generator.generate_instructions(sample_instructions, seed_type, instruction_generation_prompt, 1)

print("Newly generated instructions:")
for instruction in new_instructions:
    print("-", instruction)

# Produce JSON equivalent of the the newly generated instructions
instruction_json_pairs, difficulty = json_generator.generate_json(new_instructions, data_sample, seed_type, json_generation_prompt)

print("Newly generated JSONs:")
for pair in instruction_json_pairs:
    print("-", pair)

# Validate Logic using the API
validated_pairs, difficulty = logic_validator.validate_logic(instruction_json_pairs, seed_type, logic_validation_prompt)

print("Validated JSONs:")
for pair in validated_pairs:
    print("-", pair)

/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "InstructionJSONPair" shadows an attribute in parent "BaseModel"
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "ValidatedSolution" shadows an attribute in parent "BaseModel"
  warnings.warn(


Newly generated instructions:
- Go to the orchard and pick up five crates of harvested apples. Proceed to the packaging unit and wait until the crates are unloaded. Once unloaded, travel to the recharge station to charge for 10 minutes before the next task.
- Head towards the logistics center and collect a pallet of auto parts. Deliver the pallet to the shipping dock. After completing the delivery, wait for any return shipments for 10 minutes. If none arrive, return to the central depot.
- Navigate to the fertilizer storage area to collect three bags of fertilizer. Transport the fertilizer bags to the greenhouse facility and wait until they are unloaded. Once completed, proceed to the equipment bay for regular maintenance checks.


/usr/local/lib/python3.10/dist-packages/pydantic/json_schema.py:2279: PydanticJsonSchemaWarning: Default value <function BaseModel.json at 0x7fe7e99712d0> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


Newly generated JSONs:
- ('Go to the orchard and pick up five crates of harvested apples. Proceed to the packaging unit and wait until the crates are unloaded. Once unloaded, travel to the recharge station to charge for 10 minutes before the next task.', {'instruction': [{'action': 'go_to', 'location': 'orchard', 'object': None, 'duration': None}, {'action': 'wait_for_payload_ready', 'location': 'orchard', 'object': [{'description': 'crates of harvested apples', 'num': '5'}], 'duration': None}, {'action': 'go_to', 'location': 'packaging unit', 'object': None, 'duration': None}, {'action': 'wait_for_payload_removal', 'location': 'packaging unit', 'object': [{'description': 'crates of harvested apples', 'num': '5'}], 'duration': None}, {'action': 'go_to', 'location': 'recharge station', 'object': None, 'duration': None}, {'action': 'wait_for_seconds', 'location': 'recharge station', 'object': None, 'duration': 600}]})
- ('Head towards the logistics center and collect a pallet of auto par